# GDP vs. Life Expectancy Global Bubble Analysis

Gapminder-style bubble charts are powerful visualizations mapping wealth (GDP per capita) on the horizontal axis against health (life expectancy) on the vertical axis, with country populations scaling the bubble dimensions. This notebook simulates demographic indices across 150 countries grouped into four global regions and visualizes indicators using an interactive Plotly bubble chart.



In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Generate synthetic global country indexes (150 countries)
np.random.seed(72)
n_countries = 150

regions = np.random.choice(['Americas', 'Europe', 'Asia-Pacific', 'Africa'], size=n_countries, p=[0.25, 0.25, 0.3, 0.2])

# Construct coordinates correlated to regions (e.g. Europe has higher GDP & Life Expectancy)
gdp_per_capita = []
life_expectancy = []
population = []

for region in regions:
    if region == 'Europe':
        gdp = np.random.lognormal(mean=10.2, sigma=0.4) # High GDP
        life = np.random.normal(79, 3)
        pop = np.random.uniform(5, 120) * 1e6
    elif region == 'Americas':
        gdp = np.random.lognormal(mean=9.5, sigma=0.6)
        life = np.random.normal(75, 4)
        pop = np.random.uniform(2, 350) * 1e6
    elif region == 'Asia-Pacific':
        gdp = np.random.lognormal(mean=9.0, sigma=0.8)
        life = np.random.normal(72, 5)
        pop = np.random.uniform(5, 1400) * 1e6 # Higher population scale
    else: # Africa
        gdp = np.random.lognormal(mean=7.8, sigma=0.5)
        life = np.random.normal(63, 6)
        pop = np.random.uniform(2, 200) * 1e6
        
    gdp_per_capita.append(gdp)
    life_expectancy.append(life)
    population.append(pop)

df_global = pd.DataFrame({
    'Country': [f"Country_{i+1:03d}" for i in range(n_countries)],
    'Region': regions,
    'GDP_per_Capita': gdp_per_capita,
    'Life_Expectancy': np.clip(life_expectancy, 45, 90),
    'Population': population
})

df_global.head(10)



,Country,Region,GDP_per_Capita,Life_Expectancy,Population
0,Country_001,Americas,9057.400342,70.218205,1.414103e+08
1,Country_002,Asia-Pacific,10942.521895,76.966426,6.991156e+08
2,Country_003,Asia-Pacific,16108.019234,79.409516,1.319386e+09
3,Country_004,Europe,24795.425333,80.771540,4.544862e+07
4,Country_005,Europe,36708.261512,74.068950,4.253210e+07
5,Country_006,Asia-Pacific,4913.448366,75.700448,9.682777e+08
6,Country_007,Asia-Pacific,5648.502846,64.876573,9.530026e+08
7,Country_008,Americas,10261.384389,68.999165,3.244341e+08
8,Country_009,Americas,36030.134664,73.004763,2.508123e+08
9,Country_010,Asia-Pacific,5686.131764,70.208508,1.284360e+09


## Regional Summaries

We calculate weighted average life expectancies and total populations per region to outline global indicators profiles.



In [2]:
df_regional = df_global.groupby('Region').agg(
    Avg_GDP=('GDP_per_Capita', 'mean'),
    Avg_Life_Exp=('Life_Expectancy', 'mean'),
    Total_Population=('Population', 'sum')
)
print("Regional Aggregations Summary:")
df_regional



Regional Aggregations Summary:


,Avg_GDP,Avg_Life_Exp,Total_Population
Region,,,
Africa,2776.182318,62.760902,2.610957e+09
Americas,15582.603313,74.722222,7.018004e+09
Asia-Pacific,10635.825836,72.592043,3.797663e+10
Europe,29604.874716,79.338169,2.223752e+09


## Interactive Global Bubble Chart

Using Plotly, we construct our Gapminder-style bubble chart, mapping regions to distinct categorical colors and population vectors to bubble sizes.



In [3]:
# Convert structures to standard lists for JSON serialization compatibility
regions_list = ['Americas', 'Europe', 'Asia-Pacific', 'Africa']
region_colors = {
    'Americas': '#3B82F6',
    'Europe': '#10B981',
    'Asia-Pacific': '#F59E0B',
    'Africa': '#EF4444'
}

fig = go.Figure()

for reg in regions_list:
    df_reg = df_global[df_global['Region'] == reg]
    
    # Calculate bubble sizing based on population square-roots
    sizes = np.sqrt(df_reg['Population'] / 1e6) * 3 + 6
    
    fig.add_trace(go.Scatter(
        x=list(df_reg['GDP_per_Capita']),
        y=list(df_reg['Life_Expectancy']),
        mode='markers',
        name=reg,
        text=list(df_reg['Country']),
        marker=dict(
            size=list(sizes),
            color=region_colors[reg],
            opacity=0.75,
            line=dict(width=1, color='white')
        ),
        hoverinfo='text',
        hovertext=[
            f"<b>{row['Country']}</b><br>Region: {row['Region']}<br>GDP/Capita: ${row['GDP_per_Capita']:,.0f}<br>Life Expectancy: {row['Life_Expectancy']:.1f} Yrs<br>Population: {row['Population']/1e6:.1f}M"
            for _, row in df_reg.iterrows()
        ]
    ))

fig.update_layout(
    title='Global Wealth vs. Health: GDP per Capita vs. Life Expectancy (2025)',
    xaxis=dict(
        title='GDP per Capita ($ USD, Log Scale)',
        type='log',
        gridcolor='#F1F5F9'
    ),
    yaxis=dict(
        title='Life Expectancy at Birth (Years)',
        gridcolor='#F1F5F9'
    ),
    template='plotly_white',
    width=650,
    height=480,
    legend_title='Global Region'
)

fig.show()
